In [ ]:
import sys, os, glob
import SimpleITK as sitk
#加载pydicom
import pydicom
import numpy as np

#函数主要处理DICOM数据
def load_dicom(slice_list):
    #print(slice_list)
    slices = []#初始化空列表，用于存储处理后的DICOM格式的切片
    #切片遍历，读取相邻的DICOM文件
    for s, t in zip(slice_list,slice_list[1:]):
        slice = pydicom.read_file(s)
        slice2 = pydicom.read_file(t)
        #修复图像方向问题，确保一致性
        if slice.ImageOrientationPatient==[0,1,0,0,0,-1]:#错误方向
            slice.ImageOrientationPatient=[1,0,0,0,1,0]#修正为标准方向
            slice2.ImageOrientationPatient=[1,0,0,0,1,0]
       #切片筛选逻辑
       #条件1：切片厚度与实际位置差匹配，接受切片
        if float(slice.SliceThickness) == np.abs(slice.ImagePositionPatient[2] - slice2.ImagePositionPatient[2]):#用ImagePositionPatient计算实际间距
            slices.append(slice)
        #条件2：方向与标准方向一致，接受切片
        elif slice.ImageOrientationPatient==[1,0,0,0,1,0]:    
            slices.append(slice)
        if t==slice_list[-1]:
            slices.append(slice2)
    try:
        # seriesDesc = slices[0].SeriesDescription
        slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
        #切片厚度计算，多层验证确保切片厚度准确性
        try:
            slice_thickness = np.abs(slices[0].ImagePositionPatient[2] - slices[1].ImagePositionPatient[2])
            #如果厚度异常（>3mm或为0），使用第9-10切片重新计算
            if slice_thickness > 3 or slice_thickness == 0:
                slice_thickness = np.abs(slices[9].ImagePositionPatient[2] - slices[10].ImagePositionPatient[2])
        #备选方法：使用SliceLocation属性
        except:
            slice_thickness = np.abs(slices[0].SliceLocation - slices[1].SliceLocation)
            if slice_thickness > 3:
                slice_thickness = np.abs(slices[9].SliceLocation - slices[10].SliceLocation)
    except Exception as e:
        print (e)
        print('No position found for image', slice_list[0])
        if 'Sinai'  or 'TCGA' or 'E3311' in slice_list[0]:
            slice_thickness = float(slices[0].SliceThickness)
            print("fall back slice thickness: ", slice_thickness)
        else:
            #slice_thickness = np.abs(slices[0].SliceLocation - slices[1].SliceLocation)
            #print("slice thickness: ", slice_thickness)
            return []
    #统一所有切片的厚度值
    for s in slices:
        s.SliceThickness = slice_thickness
        #构建完整的空间信息
        print('image position:', s.ImagePositionPatient)
        # [像素间距X, 像素间距Y, 切片厚度]
    img_spacing = [float(slices[0].PixelSpacing[0]),float(slices[0].PixelSpacing[1]), slice_thickness]
    #print('img_spacing: ', img_spacing)
    #图像方向余弦矩阵
    img_direction = [float(i) for i in slices[0].ImageOrientationPatient] + [0, 0, 1]
    #图像起始位置
    img_origin = slices[0].ImagePositionPatient
    
    return slices, img_spacing, img_direction, img_origin

# PET扫描的DICOM数据处理
def load_dicom_pet(slice_list):
    #直接列表推导式读取所有切片，没有复杂的相邻切片处理
    slices = [pydicom.read_file(s) for s in slice_list]
    #排序和厚度计算，相比CT版本，没有9-10切片的异常检查
    try:
        # seriesDesc = slices[0].SeriesDescription，按Z轴位置排序切片
        slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
        try:
            #首选：用前两片的ImagePositionPatient计算
            slice_thickness = np.abs(slices[0].ImagePositionPatient[2] - slices[1].ImagePositionPatient[2])
        except:
            #备选：使用SliceLocation计算
            slice_thickness = np.abs(slices[0].SliceLocation - slices[1].SliceLocation)
    #异常处理，捕获异常并打印错误信息，产生空列表
    except Exception as e:
        print (e)
        #print 'No position found for image', slice_list[0]
        return []
    #元数据处理和返回
    for s in slices:
        s.SliceThickness = slice_thickness#统一所有切片的厚度值
        #[像素间距X, 像素间距Y, 切片厚度]
    img_spacing = [float(slices[0].PixelSpacing[0]),float(slices[0].PixelSpacing[1]), slice_thickness]
    img_direction = [int(i) for i in slices[0].ImageOrientationPatient] + [0, 0, 1]# 图像方向
    img_origin = slices[0].ImagePositionPatient#图像起始位置
    return slices, img_spacing, img_direction, img_origin
    
#将原始的DICOM像素数组转换为具有物理意义的Hounsfield单位数组。    
def getPixelArray(slices):
    #创建图像堆栈
    image = np.stack([s.pixel_array for s in slices])#将多个DICOM切片的像素数组合并为三维数组
    image = image.astype(np.int16)                  #转换为int16类型以确保数据一致性和内存效率
    #像素值转换处理
    #DICOM原始像素值需要经过线性变换得到有物理意义的数值
    #实际值 = 原始像素值 × （重缩放斜率）RescaleSlope + （重缩放截距）RescaleIntercept
    for slice_number in range(len(slices)):
        intercept = slices[slice_number].RescaleIntercept
        slope = slices[slice_number].RescaleSlope
        #如果斜率不为1，先进行乘法运算
        if slope != 1:
            image[slice_number] = slope * image[slice_number].astype(np.float64)
            image[slice_number] = image[slice_number].astype(np.int16)
        image[slice_number] += np.int16(intercept)#对所有切片加上截距值
    return np.array(image, dtype=np.int16)#确保返回统一数据类型的数组
#核心处理流程
def run_core(dicom_dir, image_format):
    print ('Processing patient ', dicom_dir)
    #智能去重：通过数字提取检测并移除重复切片。
    dicomFiles = sorted(glob.glob(dicom_dir + '/[!D]*'))
    dicomCheck = list(map(lambda sub:int(''.join([i for i in sub if i.isnumeric()])), dicomFiles)) #Extracts only the numbers to check for duplicates
    if len(dicomCheck) > len(set(dicomCheck)):
        dicomFiles = [item for item in dicomFiles if '.dcm' not in item]
        print("Removing duplicate slices")
    if image_format=='ct':
        slices, img_spacing, img_direction, img_origin = load_dicom(dicomFiles)
        print('img_spacing: ', img_spacing)
    elif image_format=='pet':
        slices, img_spacing, img_direction, img_origin = load_dicom_pet(dicomFiles)
    if 0.0 in img_spacing:
        print ('ERROR - Zero spacing found for patient,', img_spacing)
        return ''
    
    
    if image_format=='ct':
        imgCube = getPixelArray(slices)
    elif image_format=='pet':
        imgCube = getPixelArray_pet(slices, image_format)   
    # 构建完整的SimpleITK图像对象，包含所有必要的元数据。
    imgSitk = sitk.GetImageFromArray(imgCube)
    imgSitk.SetSpacing(img_spacing)
    imgSitk.SetDirection(img_direction)
    imgSitk.SetOrigin(img_origin)
    
    return imgSitk
    #except:
def dcm_to_nrrd(dataset, patient_id, data_type, input_dir, output_dir, image_format, save=True):
    """
    Converts a stack of slices into a single .nrrd file and saves it.
    Args:
        dataset (str): Name of dataset.
        patient_id (str): Unique patient id.
        data_type (str): Type of data (e.g., ct, pet, mri..)
        input_dir (str): Path to folder containing slices.
        output_dir (str): Path to folder where nrrd will be saved.
        save (bool): If True, the nrrd file is saved
    Returns:
        The sitk image object.
    Raises:
        Exception if an error occurs.
    """
    try:
        nrrd_name = "{}_{}_{}_raw_raw_raw_xx.nrrd".format(dataset, patient_id, data_type)
        nrrd_file_path = os.path.join(output_dir, nrrd_name)
        sitk_object = run_core(input_dir, image_format)
        if save:
            #使用压缩减少存储空间。
            nrrdWriter = sitk.ImageFileWriter()
            nrrdWriter.SetFileName(nrrd_file_path)
            nrrdWriter.SetUseCompression(True)
            nrrdWriter.Execute(sitk_object)
        print ("dataset:{} patient_id:{} done!".format(dataset, patient_id))
        return sitk_object
    except Exception as e:
        print ("dataset:{} patient_id:{} error:{}".format(dataset, patient_id, e))